# 1 循环神经网络

In [1]:
import torch
import matplotlib.pyplot as plt

In [3]:
X, W_xh = torch.normal(0, 1, (3, 1)), torch.normal(0, 1, (1, 4))
H, W_hh = torch.normal(0, 1, (3, 4)), torch.normal(0, 1, (4, 4))
X @ W_xh + H @ W_hh

tensor([[ 0.1181,  2.3764,  0.6635,  5.8496],
        [ 0.5220,  0.2261, -0.2766,  0.3094],
        [-2.2483, -0.0957,  0.5941, -2.6135]])

In [4]:
torch.cat((X, H), dim=1) @ torch.cat((W_xh, W_hh), dim=0)

tensor([[ 0.1181,  2.3764,  0.6635,  5.8496],
        [ 0.5220,  0.2261, -0.2766,  0.3094],
        [-2.2483, -0.0957,  0.5941, -2.6135]])

## 2 循环神经网络的实现

In [12]:
from torch import nn
from torch.nn import functional as F
import collections
import random
import re

In [46]:
def read_time_machine():
    ''' 读取时间机器数据集 '''
    with open('time_machine.txt', 'r', encoding='utf-8') as f:
        lines = f.readlines()
    return [re.sub('[^A-Za-z]+', ' ', line).strip().lower() for line in lines]

def tokenize(lines, token='work'):
    ''' 对时间机器数据集进行分词 '''
    if token == 'work':
        return [line.split() for line in lines]
    elif token == 'char':
        return [list(line) for line in lines]
    else:
        print('错误：未知token类型' + token)

class Vocab:
    """词汇表"""
    def __init__(self, tokens=None, min_freq=0, reserved_tokens=None):
        if tokens is None:
            tokens = []
        if reserved_tokens is None:
            reserved_tokens = []
        counter = count_corpus(tokens)
        self._token_freqs = sorted(counter.items(), key=lambda x: x[1], reverse=True)
        self.idx_to_token = ['<unk>'] + reserved_tokens
        self.token_to_idx = {token: idx for idx, token in enumerate(self.idx_to_token)}
        for token, freq in self._token_freqs:
            if freq < min_freq:
                break
            if token not in self.idx_to_token:
                self.idx_to_token.append(token)
                self.token_to_idx[token] = len(self.idx_to_token) - 1

    def __len__(self):
        return len(self.idx_to_token)
        
    def __getitem__(self, tokens):
        if not isinstance(tokens, (list, tuple)):
            return self.token_to_idx.get(tokens, self.unk)
        return [self.__getitem__(token) for token in tokens]
        
    def to_tokens(self, indices):
        if not isinstance(indices, (list, tuple)):
            return self.idx_to_token[indices]
        return [self.idx_to_token[idx] for idx in indices]
        
    @property
    def unk(self):
        return 0
        
    @property
    def token_freqs(self):
        return self._token_freqs
        
def count_corpus(tokens):
    """统计文本中的token出现频率"""
    if len(tokens) == 0 or isinstance(tokens[0], list):
        tokens=[token for line in tokens for token in line]
    return collections.Counter(tokens)

def load_corpus_time_machine(max_tokens=-1):
    ''' 加载时间机器数据集 '''
    lines = read_time_machine()
    tokens = tokenize(lines, 'char')
    vocab = Vocab(tokens)
    corpus = [vocab[token] for line in tokens for token in line]
    if max_tokens > 0:
        corpus = corpus[:max_tokens]
    return corpus, vocab

def seq_data_iter_random(corpus, batch_size, num_steps):
    """随机迭代序列数据"""
    corpus = corpus[random.randint(0, num_steps - 1):]
    num_subseqs = (len(corpus) - 1) // num_steps
    initial_indices = list(range(0, num_subseqs * num_steps, num_steps))
    random.shuffle(initial_indices)

    def data(pos):
        return corpus[pos:pos + num_steps]
    
    num_batches = num_subseqs // batch_size
    for i in range(0, batch_size * num_batches, batch_size):
        initial_indices_per_batch = initial_indices[i: i + batch_size]
        X = [data(j) for j in initial_indices_per_batch]
        Y = [data(j + 1) for j in initial_indices_per_batch]
        yield torch.tensor(X), torch.tensor(Y)

class SeqDataLoader:
    ''' 序列数据加载器 '''
    def __init__(self, batch_size, num_steps, max_tokens):
        self.data_iter_fn = seq_data_iter_random
        self.corpus, self.vocab = load_corpus_time_machine(max_tokens)
        self.batch_size = batch_size
        self.num_steps = num_steps

    def __iter__(self):
        return self.data_iter_fn(self.corpus, self.batch_size, self.num_steps)
    
def load_data_time_machine(batch_size, num_steps, max_tokens=10000):
    data_iter = SeqDataLoader(batch_size, num_steps, max_tokens)
    return data_iter, data_iter.vocab

In [47]:
batch_size, num_steps = 32, 35
train_iter, vocab = load_data_time_machine(batch_size, num_steps)

## 2.1 独热编码

In [48]:
F.one_hot(torch.tensor([0, 2]), len(vocab))

tensor([[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0]])

In [49]:
X = torch.arange(10).reshape((2, 5))
F.one_hot(X.T, 28).shape

torch.Size([5, 2, 28])

## 2.2 初始化模型参数

In [50]:
def get_params(vocab_size, num_hiddens, device):
    num_inputs = num_outputs = vocab_size

    def normal(shape):
        return torch.randn(size=shape, device=device) * 0.01
    
    W_xh = normal((num_inputs, num_hiddens))
    W_hh = normal((num_hiddens, num_hiddens))
    b_h = torch.zeros(num_hiddens, device=device)
    W_hp = normal((num_hiddens, num_outputs))
    b_p = torch.zeros(num_outputs, device=device)
    params = [W_xh, W_hh, b_h, W_hp, b_p]
    for param in params:
        param.requires_grad = True
    return params

## 2.3 循环神经网络模型

In [51]:
def init_rnn_state(batch_size, num_hiddens, device):
    ''' 初始化循环神经网络状态 '''
    return (torch.zeros((batch_size, num_hiddens), device=device), )


In [52]:
def rnn(input, state, params):
    ''' 循环神经网络模型 '''
    W_xh, W_hh, b_h, W_hp, b_p = params
    H, = state
    outputs = []
    for X in input:
        H = torch.tanh(X @ W_xh + H @ W_hh + b_h)
        Y = H @ W_hp + b_p
        outputs.append(Y)
    return torch.cat(outputs, dim=0), (H,)

In [53]:
class RNNModelScratch:
    def __init__(self, vocab_size, num_hiddens, device, get_params, init_state, forward_fn):
        self.vocab_size, self.num_hiddens = vocab_size, num_hiddens
        self.params = get_params(vocab_size, num_hiddens, device)
        self.init_state, self.forward_fn = init_state, forward_fn

    def __call__(self, X, state):
        X = F.one_hot(X.T, self.vocab_size).type(torch.float32)
        return self.forward_fn(X, state, self.params)
        
    def begin_state(self, batch_size, device):
        return self.init_state(batch_size, self.num_hiddens, device)

In [54]:
num_hiddens = 512
device = torch.device('cuda')
X = X.to(device)
net = RNNModelScratch(len(vocab), num_hiddens, device, get_params, init_rnn_state, rnn)
state = net.begin_state(X.shape[0], device)
Y, new_state = net(X, state)
Y.shape, len(new_state), new_state[0].shape

(torch.Size([10, 28]), 1, torch.Size([2, 512]))

## 2.4 预测

In [76]:
def predict(prefix, num_preds, net, vocab, device):
    ''' 在prefix后预测序列 '''
    state = net.begin_state(batch_size=1, device=device)
    outputs = [vocab[prefix[0]]]
    get_input = lambda: torch.tensor([outputs[-1]], device=device).reshape((1, 1))
    for y in prefix[1:]:
        _, state = net(get_input(), state)
        outputs.append(vocab[y])
    for _ in range(num_preds):
        y, state = net(get_input(), state)
        outputs.append(int(y.argmax(dim=1).reshape(1)))
    return ''.join([vocab.idx_to_token[i] for i in outputs])

In [77]:
predict('time traveller', 10, net, vocab, device)

'time travelleroon lnptlu'

## 2.5 梯度截断

In [57]:
def grad_clipping(net, theta):
    if isinstance(net, nn.Module):
        params = [p for p in net.parameters() if p.requires_grad]
    else: 
        params = net.params
    norm = torch.sqrt(sum(torch.sum(p.grad ** 2) for p in params))
    if norm > theta:
        for param in params:
            param.grad[:] *= theta / norm

## 2.6 训练

In [58]:
import math

def train_epoch(net, train_iter, loss, updater, device):
    ''' 训练模型一个epoch '''
    state = None
    loss_num, num = 0, 0
    for X, Y in train_iter:
        if state is None:
            state = net.begin_state(batch_size=X.shape[0], device=device)
        else:
            if isinstance(net, nn.Module) and not isinstance(state, tuple):
                state.detach_()
            else:
                for s in state:
                    s.detach_()
        y = Y.T.reshape(-1)
        X, y = X.to(device), y.to(device)
        y_hat, state = net(X, state)
        l = loss(y_hat, y)
        if isinstance(updater, torch.optim.Optimizer):
            updater.zero_grad()
            l.backward()
            grad_clipping(net, 1)
            updater.step()
        else:
            l.backward()
            grad_clipping(net, 1)
            updater(batch_size=1)
        loss_num += l.item() * y.numel()
        num += y.numel()
    return math.exp(loss_num / num)


In [65]:
def train(net, train_iter, vocab, lr, epochs, device):
    ''' 训练模型 '''
    loss = nn.CrossEntropyLoss()
    if isinstance(net, nn.Module):
        updater = torch.optim.SGD(net.parameters(), lr=lr)
    else:
        updater = lambda batch_size: sgd(net.params, lr, batch_size)
    pred = lambda prefix: predict(prefix, 50, net, vocab, device)

    for epoch in range(epochs):
        l = train_epoch(net, train_iter, loss, updater, device)
    print(f'困惑度：{l:.2f}')
    print(pred('time traveller'))
    print(pred('traveller'))

In [66]:
def sgd(params, lr, batch_size):
    ''' 手动实现SGD更新参数 '''
    for param in params:
        grad = param.grad.data * lr / batch_size
        param.data -= grad
        param.grad.data.zero_()

In [67]:
epochs, lr = 500, 1
train(net, train_iter, vocab, lr, epochs, device)

困惑度：1.42
time traveller smiled round at us then still smiling faintly and
traveller smiled round at us then still smiling faintly and


# 3. 简洁实现

In [68]:
batch_size, num_steps = 32, 35
train_iter, vocab = load_data_time_machine(batch_size, num_steps)

## 3.1 定义模型

In [83]:
num_hiddens = 512
rnn_layer = nn.RNN(len(vocab), num_hiddens)

In [70]:
state = torch.zeros((1, batch_size, num_hiddens))
state.shape

torch.Size([1, 32, 256])

In [71]:
X = torch.rand(size=(num_steps, batch_size, len(vocab)))
Y, state_new = rnn_layer(X, state)
Y.shape, state_new.shape

(torch.Size([35, 32, 256]), torch.Size([1, 32, 256]))

In [74]:
class RNNModel(nn.Module):
    def __init__(self, rnn_layer, vocab_size, **kwargs):
        super().__init__(**kwargs)
        self.rnn = rnn_layer
        self.vocab_size = vocab_size
        self.num_hiddens = rnn_layer.hidden_size
        if not self.rnn.bidirectional:
            self.num_directions = 1
            self.linear = nn.Linear(self.num_hiddens, self.vocab_size)
        else:
            self.num_directions = 2
            self.linear = nn.Linear(self.num_hiddens * 2, self.vocab_size)

    def forward(self, inputs, state):
        X = F.one_hot(inputs.T.long(), self.vocab_size)
        X = X.to(torch.float32)
        Y, state = self.rnn(X, state)
        output = self.linear(Y.reshape((-1, Y.shape[-1])))
        return output, state
    
    def begin_state(self, device, batch_size=1):
        if not isinstance(self.rnn, nn.LSTM):
            return torch.zeros((self.num_directions * self.rnn.num_layers, batch_size, self.num_hiddens), device=device)
        else:
            return (torch.zeros((self.num_directions * self.rnn.num_layers, batch_size, self.num_hiddens), device=device), 
                    torch.zeros((self.num_directions * self.rnn.num_layers, batch_size, self.num_hiddens), device=device))

## 3.2 训练与预测

In [84]:
device = torch.device('cuda')
net = RNNModel(rnn_layer, vocab_size=len(vocab))
net = net.to(device)
predict('time traveller', 10, net, vocab, device)

'time traveller  x n n n '

In [85]:
epochs, lr = 2000, 1
train(net, train_iter, vocab, lr, epochs, device)

困惑度：1.44
time traveller smiled round at us then still smiling faintly and
traveller smiled round at us then still smiling faintly and
